## Assignment 4.

## 1. Objectives & Setup
The goals of this assignment are:
1. To load and work with pretrained GloVe word embeddings.
2. To explore semantic relationships in vector space using vector arithmetic and **cosine similarity**.

In [2]:
# %pip install gensim

In [ ]:
import numpy as np
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

# Load the GloVe file
glove_model = KeyedVectors.load_word2vec_format("glove.6B.100d.txt", binary=True)

embedding_weights = glove_model.vectors
vocabulary = glove_model.index_to_key
index_word = {idx: word for idx, word in enumerate(vocabulary)}
word_index = {word: idx for idx, word in enumerate(vocabulary)}

print("Embedding matrix shape:", embedding_weights.shape)

Embedding matrix shape: (400000, 100)


In [4]:
# Function to find nearest words to any given vector (adapted from course material)
def find_nearest_to_vector(target_vector, top_n=10, exclude_words=None):
    if exclude_words is None:
        exclude_words = []
        
    target_embedding = target_vector.reshape(1, -1)  # shape (1, 100)
    similarities = cosine_similarity(embedding_weights, target_embedding).reshape(-1)
    
    # Sort indices from highest similarity to lowest
    similar_indices = np.argsort(similarities)[::-1]
    
    results = []
    for idx in similar_indices:
        word = index_word[idx]
        if word not in exclude_words:
            results.append((word, float(similarities[idx])))
        if len(results) == top_n:
            break
    return results

# Fetch word vectors for "woman", "man", and "king"
vec_woman = embedding_weights[word_index["woman"]]
vec_man = embedding_weights[word_index["man"]]
vec_king = embedding_weights[word_index["king"]]

# Calculate: vec("woman") - vec("man") + vec("king")
result_vec = vec_woman - vec_man + vec_king

print("Nearest words (including input words):")
print(find_nearest_to_vector(result_vec, top_n=5))

print("\nNearest words (excluding input words):")
print(find_nearest_to_vector(result_vec, top_n=5, exclude_words=["woman", "man", "king"]))

Nearest words (including input words):
[('king', 0.8551836609840393), ('queen', 0.7834413051605225), ('monarch', 0.6933802366256714), ('throne', 0.6833109855651855), ('daughter', 0.6809082627296448)]

Nearest words (excluding input words):
[('queen', 0.7834413051605225), ('monarch', 0.6933802366256714), ('throne', 0.6833109855651855), ('daughter', 0.6809082627296448), ('prince', 0.6713141798973083)]


### Explanation of the Result

- **Unfiltered results:** When the input words are included, `king` is the closest match (cosine similarity 0.855), followed by `queen` (0.783). This happens because `vec("woman") - vec("man")` is a relatively small offset, so the result vector stays very close to `vec("king")`. For this reason, analogy queries conventionally exclude the input words.
- **Filtered results:** With `woman`, `man` and `king` excluded, the nearest word is **`queen`** (0.783), followed by `monarch` (0.693), `throne` (0.683), `daughter` (0.681) and `prince` (0.671). Most neighbours are royalty-related, but `daughter` shows that the "female" direction also pulls in family terms, not just royal ones. The result is approximate: `queen` is the best match, but the vector is not an exact hit.

#### Conclusion
**GloVe** learns word vectors from global word-word co-occurrence statistics. `man` and `woman` appear in very similar contexts, differing mainly in gendered usage, so `vec("woman") - vec("man")` roughly isolates a female-vs-male direction. Adding that direction to `vec("king")` moves the vector toward the region of female royalty, where `queen` sits. Cosine similarity compares only the direction of vectors, not their length, so the nearest word is the one pointing in the most similar direction.

## 2. Additional Findings

In [5]:
def test_analogy(word_a, word_b, word_c, top_n=5):
    """Computes vec(word_b) - vec(word_a) + vec(word_c) and prints nearest words."""
    vec_a = embedding_weights[word_index[word_a]]
    vec_b = embedding_weights[word_index[word_b]]
    vec_c = embedding_weights[word_index[word_c]]
    
    target = vec_b - vec_a + vec_c
    nearest = find_nearest_to_vector(target, top_n=top_n, exclude_words=[word_a, word_b, word_c])
    
    print(f'"{word_a}" -> "{word_b}"  |  "{word_c}" -> ?')
    for word, sim in nearest:
        print(f"   {word:<15} (cosine similarity: {sim:.4f})")
    print()

# Country -> Capital
test_analogy("france", "paris", "finland")

# Country -> Currency
test_analogy("usa", "dollar", "japan")

# Present tense -> Past tense
test_analogy("walk", "walked", "swim")

# Adjective -> Comparative
test_analogy("slow", "slower", "fast")

# Occupational gender bias example
test_analogy("man", "doctor", "woman")

"france" -> "paris"  |  "finland" -> ?
   helsinki        (cosine similarity: 0.8306)
   stockholm       (cosine similarity: 0.7115)
   oslo            (cosine similarity: 0.6272)
   budapest        (cosine similarity: 0.6113)
   copenhagen      (cosine similarity: 0.6034)

"usa" -> "dollar"  |  "japan" -> ?
   yen             (cosine similarity: 0.7021)
   japanese        (cosine similarity: 0.6807)
   currency        (cosine similarity: 0.6787)
   tokyo           (cosine similarity: 0.6611)
   greenback       (cosine similarity: 0.6384)

"walk" -> "walked"  |  "swim" -> ?
   swam            (cosine similarity: 0.7515)
   swimming        (cosine similarity: 0.6240)
   drowned         (cosine similarity: 0.5790)
   overboard       (cosine similarity: 0.5782)
   raced           (cosine similarity: 0.5580)

"slow" -> "slower"  |  "fast" -> ?
   faster          (cosine similarity: 0.8030)
   quicker         (cosine similarity: 0.6689)
   pace            (cosine similarity: 0.6581)
   fast

### Explanation of Additional Findings

`test_analogy(a, b, c)` computes `vec(b) - vec(a) + vec(c)`. This means basically: **"a is to b as c is to ?"**, and excludes the three input words from the results.

1. **Geographic and cultural knowledge.**
   - `france -> paris | finland -> ?` returns `helsinki` (around 0.831), with other Nordic and European capitals (`stockholm`, `oslo`, `budapest`, `copenhagen`) next. The country-to-capital offset transfers well here, and the runners-up are semantically close capitals.
   - `usa -> dollar | japan -> ?` returns `yen` (0.702), but the result is noisier: `japanese` (0.681), `currency` (0.679) and `tokyo` (0.661) follow closely. The vector captures "money" and "Japan" associations in general, not only the exact currency relation.

2. **Syntactic and grammatical regularities.**
   - `walk -> walked | swim -> ?` returns the irregular past tense `swam` (0.752). Lower-ranked results such as `drowned` and `overboard` reflect typical contexts of "swim" in text, which shows that the embeddings encode co-occurrence patterns rather than explicit grammar rules.
   - `slow -> slower | fast -> ?` returns `faster` (0.803), followed by `quicker`, `pace`, `fastest` and `speeds`. The comparative offset works, though related words about speed also appear.

3. **Societal bias.** `man -> doctor | woman -> ?` returns `nurse` (0.776) first, then `physician` (0.713), `doctors` (0.679), `pregnant` (0.679) and `patient` (0.677).
   - The nearest non-input word is `nurse`, a stereotypically female-coded occupation. The input word `doctor` is excluded, so it cannot appear in the results.
   - `physician` is a close second, so the result is not purely stereotyped. The appearance of `pregnant`, however, shows that the "woman" direction brings in gendered associations that have nothing to do with the profession.
   - GloVe is trained on Wikipedia and Gigaword news text, so it reflects the gender imbalances and stereotypes of that data.

#### Overall

The analogies work well for several relation types, but they are approximate. Rankings are often close, and related words can outrank the "correct" answer.